# 02 · Why DETR? From anchors and NMS to set prediction

> **Paper:** *End-to-End Object Detection with Transformers* — Carion et al., 2020 · §1 (Introduction), Fig. 1
> **Code:** [`models/detr.py`](../models/detr.py)

**What you'll do in this notebook**
1. See what object detection looked like *before* DETR, and why it was messy.
2. Load the pretrained DETR-R50 and detect objects in a real photo.
3. Verify, by searching the source, that **there is no NMS anywhere**.
4. Look inside all 100 predictions to see what "set prediction" actually means.

*New to PyTorch? Every tensor in this tutorial is annotated with a `# shape: (...)` comment. Read those first — shapes are the fastest way to understand a vision model. [`00 · PyTorch essentials`](00_pytorch_essentials.ipynb) covers every operation DETR uses, each one tied back to the line of `models/` that needs it.*

## 1. The problem DETR is solving

An object detector must answer: **"what objects are in this image, and where?"** The answer is a *set* — 2 cats and 2 remotes, in no particular order.

Sets are awkward for neural networks, because a network outputs a fixed-size, *ordered* tensor. Classic detectors dodge this with a three-step workaround:

| Step | What happens | Hand-designed pieces |
|---|---|---|
| 1. Propose | Blanket the image with ~100k **anchors** (Faster R-CNN) or grid cells (YOLO) | anchor scales, aspect ratios, strides |
| 2. Classify + regress | For every anchor: is there an object? nudge the box | rule for which anchor "owns" which object |
| 3. Deduplicate | ~100k boxes → a handful, via **Non-Maximum Suppression** | IoU threshold |

Every one of those hand-designed pieces is a hyperparameter someone tuned. And NMS is not even differentiable — it's a post-processing script bolted onto the end.

### DETR's answer

> *"We propose a direct set prediction approach to bypass the surrogate tasks."* — §1

DETR predicts a **fixed set of exactly N = 100 boxes, all at once, in one forward pass.** Most of them come out as a special `∅` ("no object") class. No anchors. No NMS. The pipeline is:

```
image ──► CNN backbone ──► Transformer encoder ──► Transformer decoder ──► 100 × (class, box)
                                                          ▲
                                                   100 object queries
```

The trick that makes it work is the **loss**, not the architecture: a bipartite matching (Hungarian) loss that scores a *set* against a *set*, ignoring order. That's notebook `05` — first let's just watch it run.

## 2. Setup

`detr_utils.py` (next to this notebook) puts the repo root on `sys.path` and holds the boilerplate: COCO class names, plotting, model loading.

## 0. Setup — everything this notebook needs

This notebook is **self-contained**: only third-party packages are imported, and every DETR-specific piece is written out below. Nothing comes from this repo, so you can read straight through without chasing a helper into another file.

Run this section once, then forget about it.

In [ ]:
# Standard third-party imports. Nothing from this repo -- every helper this
# notebook uses is defined below, in this file.
import itertools
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=110)
np.set_printoptions(precision=3, suppress=True)

ASSETS = "_assets"                      # downloaded images are cached here
os.makedirs(ASSETS, exist_ok=True)
print("torch", torch.__version__)

### COCO class names and plot colors

In [ ]:
# DETR predicts 91 "classes" + 1 no-object slot = 92 logits per query.
# COCO's category ids are not contiguous (they run 1..90 with gaps), so the gaps
# are filled with 'N/A' placeholders and index 0 is unused. The list MUST be
# exactly 91 long -- one short and every label after the gap is silently wrong.
COCO_CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
    'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush',
]
assert len(COCO_CLASSES) == 91, f"expected 91 classes, got {len(COCO_CLASSES)}"

COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

### Box geometry

DETR predicts boxes as **`cxcywh`** — centre + size, normalized to `[0, 1]`. IoU and plotting want **`xyxy`** corners. Mixing the two up is the single most common bug in detection code, so both conversions live here.

In [ ]:
def box_cxcywh_to_xyxy(b):
    """(cx, cy, w, h) -> (x0, y0, x1, y1), on the last dim."""
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1)


def box_xyxy_to_cxcywh(b):
    x0, y0, x1, y1 = b.unbind(-1)
    return torch.stack([(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0], dim=-1)


def box_area(b):
    """Area of (x0, y0, x1, y1) boxes."""
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    """Pairwise IoU. a: (N, 4), b: (M, 4), both xyxy. Returns (iou, union), each (N, M)."""
    area_a, area_b = box_area(a), box_area(b)
    lt = torch.max(a[:, None, :2], b[None, :, :2])          # (N, M, 2) top-left of overlap
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])          # (N, M, 2) bottom-right
    wh = (rb - lt).clamp(min=0)                             # no overlap -> 0
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union, union


def generalized_box_iou(a, b):
    """GIoU = IoU - |C \\ (A u B)| / |C|, where C is the smallest box enclosing both.

    Unlike IoU, GIoU keeps giving gradient when the boxes do not overlap at all:
    it measures how far apart they are, in units of the enclosing box.
    Range is [-1, 1] (1 = identical, -1 = infinitely far apart).
    """
    assert (a[:, 2:] >= a[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    assert (b[:, 2:] >= b[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    iou, union = box_iou(a, b)
    lt = torch.min(a[:, None, :2], b[None, :, :2])          # enclosing box
    rb = torch.max(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    enclosing = wh[:, :, 0] * wh[:, :, 1]
    return iou - (enclosing - union) / enclosing


# --- quick self-check -------------------------------------------------------
_a = torch.tensor([[0.0, 0.0, 2.0, 2.0]])
_b = torch.tensor([[1.0, 1.0, 3.0, 3.0]])
assert torch.allclose(box_iou(_a, _b)[0], torch.tensor([[1 / 7]]))        # 1 / (4+4-1)
assert torch.allclose(generalized_box_iou(_a, _a), torch.tensor([[1.0]])) # identical -> 1
_far = torch.tensor([[10.0, 10.0, 11.0, 11.0]])
assert box_iou(_a, _far)[0].item() == 0.0                                 # IoU dies...
assert generalized_box_iou(_a, _far).item() < 0                           # ...GIoU still ranks
print("box helpers ok")

### Images in, tensors out

DETR's eval transform resizes the shortest side to 800px and ImageNet-normalizes. There is no fixed crop — the model accepts any input size.

In [ ]:
SAMPLE_IMAGES = {
    "cats":    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "street":  "http://images.cocodataset.org/val2017/000000000139.jpg",
    "horses":  "http://images.cocodataset.org/val2017/000000006471.jpg",
    "kitchen": "http://images.cocodataset.org/val2017/000000002153.jpg",
}


def load_image(name_or_url):
    """Load a sample image by nickname, URL, or local path. Cached under _assets/."""
    url = SAMPLE_IMAGES.get(name_or_url, name_or_url)
    if os.path.exists(url):
        return Image.open(url).convert("RGB")
    path = os.path.join(ASSETS, os.path.basename(url))
    if not os.path.exists(path):
        with open(path, "wb") as f:
            f.write(requests.get(url, timeout=60).content)
    return Image.open(path).convert("RGB")


# DETR's eval transform: resize the shortest side to 800px, to tensor, ImageNet
# normalize. There is NO fixed crop -- DETR accepts variable input sizes.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
default_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(800),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def get_device():
    """CUDA > MPS (Apple Silicon) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

### Drawing detections

In [ ]:
def rescale_bboxes(boxes, size):
    """Normalized cxcywh in [0,1] -> absolute xyxy pixels. `size` is PIL's (W, H)."""
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(boxes)
    return b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)


def plot_results(pil_img, prob, boxes, ax=None, title=None, linewidth=2.5):
    """prob: (n, 91) softmax WITHOUT the no-object column. boxes: (n, 4) xyxy pixels."""
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(pil_img)
    for i, (p, (xmin, ymin, xmax, ymax)) in enumerate(zip(prob, boxes.tolist())):
        c = COLORS[i % len(COLORS)]
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=linewidth))
        cl = p.argmax()
        ax.text(xmin, ymin, f'{COCO_CLASSES[cl]}: {p[cl]:0.2f}', fontsize=11,
                bbox=dict(facecolor=c, alpha=0.6, edgecolor='none'), color='white')
    ax.axis('off')
    if title:
        ax.set_title(title)
    return ax

### DETR itself

Backbone, positional encoding, transformer and prediction heads, written out. This is the same architecture as [`models/`](../models) with the inference path kept.

The proof that it is faithful is `load_state_dict(...)` below: it is **strict**, so every parameter name here has to match Facebook's released checkpoint exactly or it raises.

In [ ]:
class FrozenBatchNorm2d(nn.Module):
    """BatchNorm with statistics and affine parameters frozen as plain buffers."""
    def __init__(self, n):
        super().__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, *a, **kw):
        state_dict.pop(prefix + "num_batches_tracked", None)
        super()._load_from_state_dict(state_dict, prefix, *a, **kw)

    def forward(self, x):
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        scale = w * (rv + 1e-5).rsqrt()
        return x * scale + (b - rm * scale)


class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=128, temperature=10000, scale=2 * math.pi):
        super().__init__()
        self.num_pos_feats, self.temperature, self.scale = num_pos_feats, temperature, scale

    def forward(self, x, mask):
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        eps = 1e-6
        y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
        x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale
        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)
        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[..., 0::2].sin(), pos_x[..., 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[..., 0::2].sin(), pos_y[..., 1::2].cos()), dim=4).flatten(3)
        return torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)


class Backbone(nn.Module):
    """ResNet-50 trunk with frozen BN, returning only the last (stride-32) stage."""
    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet50(weights=None, norm_layer=FrozenBatchNorm2d)
        self.body = torchvision.models._utils.IntermediateLayerGetter(net, {"layer4": "0"})
        self.num_channels = 2048

    def forward(self, x, mask):
        feat = self.body(x)["0"]
        feat_mask = F.interpolate(mask[None].float(), size=feat.shape[-2:]).to(torch.bool)[0]
        return feat, feat_mask


class TransformerEncoderLayer(nn.Module):
    """One encoder block: self-attention over image tokens, then a feed-forward net."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None, pos=None):
        # pos is added to the QUERY and the KEY but never to the VALUE:
        # position decides where to look, not what gets carried back.
        q = k = src if pos is None else src + pos
        src2 = self.self_attn(q, k, value=src, key_padding_mask=src_key_padding_mask)[0]
        src = self.norm1(src + self.dropout1(src2))
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        return self.norm2(src + self.dropout2(src2))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])

    def forward(self, src, src_key_padding_mask=None, pos=None):
        out = src
        for layer in self.layers:
            out = layer(out, src_key_padding_mask=src_key_padding_mask, pos=pos)
        return out


class TransformerDecoderLayer(nn.Module):
    """One decoder block: queries talk to each other, then to the image, then FFN."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        q = k = tgt if query_pos is None else tgt + query_pos
        tgt2 = self.self_attn(q, k, value=tgt)[0]              # queries deduplicate here
        tgt = self.norm1(tgt + self.dropout1(tgt2))
        tgt2 = self.multihead_attn(                            # queries read the image here
            query=tgt if query_pos is None else tgt + query_pos,
            key=memory if pos is None else memory + pos,
            value=memory, key_padding_mask=memory_key_padding_mask)[0]
        tgt = self.norm2(tgt + self.dropout2(tgt2))
        tgt2 = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        return self.norm3(tgt + self.dropout3(tgt2))


class TransformerDecoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        """Returns EVERY layer's output, stacked: (num_layers, num_queries, B, d_model).

        DETR keeps them all because the loss is applied after each decoder layer
        ("auxiliary decoding losses", paper section 3.2).
        """
        out = tgt
        intermediate = []
        for layer in self.layers:
            out = layer(out, memory, memory_key_padding_mask=memory_key_padding_mask,
                        pos=pos, query_pos=query_pos)
            intermediate.append(self.norm(out))
        return torch.stack(intermediate)


class Transformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.0):
        super().__init__()
        self.encoder = TransformerEncoder(d_model, nhead, dim_feedforward,
                                          num_encoder_layers, dropout)
        self.decoder = TransformerDecoder(d_model, nhead, dim_feedforward,
                                          num_decoder_layers, dropout)
        self.d_model, self.nhead = d_model, nhead

    def forward(self, src, mask, query_embed, pos_embed):
        """src/pos_embed: (B, C, H, W). mask: (B, H, W), True = padding."""
        bs, c, h, w = src.shape
        # (B, C, H, W) -> (H*W, B, C): this implementation puts the sequence axis first.
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
        mask = mask.flatten(1)

        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)
        tgt = torch.zeros_like(query_embed)                    # queries start at zero
        hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                          pos=pos_embed, query_pos=query_embed)
        return hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w)


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(nn.Linear(n, k)
                                    for n, k in zip([input_dim] + h, h + [output_dim]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < len(self.layers) - 1 else layer(x)
        return x


class DETR(nn.Module):
    def __init__(self, num_classes=91, num_queries=100, hidden_dim=256, nheads=8,
                 enc_layers=6, dec_layers=6, dim_feedforward=2048, aux_loss=False):
        super().__init__()
        self.backbone = nn.ModuleList([Backbone(), PositionEmbeddingSine(hidden_dim // 2)])
        self.transformer = Transformer(hidden_dim, nheads, enc_layers, dec_layers,
                                       dim_feedforward)
        self.input_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
        self.query_embed = nn.Embedding(num_queries, hidden_dim)
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)   # +1 = "no object"
        self.bbox_embed = MLP(hidden_dim, hidden_dim, 4, 3)
        self.num_queries = num_queries
        self.aux_loss = aux_loss                # keep every decoder layer's prediction

    def forward(self, images, mask=None):
        """images: (B, 3, H, W). mask: (B, H, W) with True on padded pixels."""
        if mask is None:                        # a single image needs no padding
            mask = torch.zeros(images.shape[0], *images.shape[-2:],
                               dtype=torch.bool, device=images.device)
        feat, feat_mask = self.backbone[0](images, mask)
        pos = self.backbone[1](feat, feat_mask)
        # hs: (num_decoder_layers, B, num_queries, hidden_dim)
        hs, memory = self.transformer(self.input_proj(feat), feat_mask,
                                      self.query_embed.weight, pos)

        outputs_class = self.class_embed(hs)             # (layers, B, queries, classes+1)
        outputs_coord = self.bbox_embed(hs).sigmoid()    # (layers, B, queries, 4)
        out = {"pred_logits": outputs_class[-1], "pred_boxes": outputs_coord[-1]}
        if self.aux_loss:
            out["aux_outputs"] = [{"pred_logits": a, "pred_boxes": b}
                                  for a, b in zip(outputs_class[:-1], outputs_coord[:-1])]
        return out


DETR_R50_URL = "https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth"


def load_pretrained_detr(device=None, aux_loss=False):
    """Build the model above and load Facebook's released COCO weights into it.

    `load_state_dict` is strict by default, which is the real test: every parameter
    name defined above has to match the official checkpoint exactly, or this raises.
    """
    model = DETR(aux_loss=aux_loss)
    ck = torch.hub.load_state_dict_from_url(DETR_R50_URL, map_location="cpu")
    model.load_state_dict(ck["model"])
    model.eval()
    return model.to(device) if device is not None else model


@torch.no_grad()
def detect(model, pil_img, threshold=0.9, device=None):
    """Returns (probs_kept (n, 91), boxes_kept xyxy pixels, raw outputs, keep mask)."""
    device = device or next(model.parameters()).device
    x = default_transform(pil_img).unsqueeze(0).to(device)
    outputs = model(x)
    probs = outputs["pred_logits"].softmax(-1)[0, :, :-1].cpu()   # drop no-object column
    keep = probs.max(-1).values > threshold
    boxes = rescale_bboxes(outputs["pred_boxes"][0, keep].cpu(), pil_img.size)
    return probs[keep], boxes, outputs, keep

In [ ]:
device = get_device()      # CUDA > MPS (Apple Silicon) > CPU
print("device:", device)

### Build the model from *this repo's* source

We deliberately do **not** use `torch.hub`. We assemble DETR out of the very modules you'll be reading — [`models/detr.py`](../models/detr.py), [`models/backbone.py`](../models/backbone.py), [`models/transformer.py`](../models/transformer.py) — and then load the official COCO weights into it. So when you edit that code, this notebook changes with it.

The checkpoint is ~159 MB and is cached in `~/.cache/torch/` after the first run.

In [ ]:
model = load_pretrained_detr(device=device)   # builds DETR, loads COCO weights, .eval()

n_params = sum(p.numel() for p in model.parameters())
n_backbone = sum(p.numel() for p in model.backbone.parameters())
n_transf = sum(p.numel() for p in model.transformer.parameters())
print(f"total parameters     : {n_params/1e6:6.1f} M")
print(f"  backbone (ResNet50): {n_backbone/1e6:6.1f} M")
print(f"  transformer        : {n_transf/1e6:6.1f} M")
print(f"  heads + embeddings : {(n_params-n_backbone-n_transf)/1e6:6.1f} M")
print()
print("num_queries (N):", model.num_queries)
print("class_embed    :", model.class_embed)   # 256 -> 91 classes + 1 no-object

Compare against the paper. Table 1 lists DETR-R50 at **41M** parameters (rounded), and §4.1 breaks that down: *"23.5M are in ResNet-50, and 17.8M in the transformer"*.

Our backbone lands on 23.45M -- that is the paper's 23.5M. Our transformer line reads lower, and the reason is a bookkeeping choice rather than a different model: we put `input_proj` on the "heads" line, the paper evidently counts it with the transformer.

| | ours | |
|---|---|---|
| transformer proper | 17.363 M | |
| `input_proj` (1x1 conv, 2048x256 + 256) | 0.525 M | |
| **sum** | **17.888 M** | -> the paper's 17.8 M |

Note also that `23.5 + 17.8 = 41.3`, not 41.5. The paper's two-part split is a breakdown of the two big blocks, not an exhaustive partition: it leaves out `query_embed` (0.026 M) and the two prediction heads (`class_embed` 0.024 M, `bbox_embed` 0.133 M). Add those and every number reconciles.

Note `class_embed` outputs **92** logits, not 91: the 91 COCO classes *plus* the `∅` no-object class. That extra slot is the whole reason DETR can emit a fixed 100 predictions for an image containing 4 objects.

## 3. Detect objects in a real image

In [ ]:
im = load_image("cats")        # a COCO val2017 image, cached in tutorial/_assets/
print("PIL size (W, H):", im.size)
im

In [ ]:
# detect() = preprocess -> forward -> softmax -> keep confident queries.
# It is defined in the setup section above; we unpack it by hand in the next cell.
probs, boxes, outputs, keep = detect(model, im, threshold=0.9)

print(f"{keep.sum().item()} of {model.num_queries} queries survived the 0.9 threshold\n")
for p, b in zip(probs, boxes):
    print(f"  {COCO_CLASSES[p.argmax()]:<10} {p.max():.3f}   xyxy={[round(v) for v in b.tolist()]}")

In [ ]:
plot_results(im, probs, boxes, title="DETR-R50 — no anchors, no NMS")
plt.show()

Two cats, two remotes, one couch. Those five boxes are the **raw, unfiltered** output of a single forward pass — nothing was suppressed or merged afterwards.

## 4. Prove it: there really is no NMS

Don't take my word for it. Search the entire detection codebase for the usual suspects.

In [ ]:
import io, pathlib, tokenize

# A plain grep would also hit comments, docstrings and printed strings -- text that
# *talks about* NMS without doing any. So tokenize each file and look only at real
# identifiers: NAME tokens. That is the honest question, "is it ever called?"
SOURCES = ([p for p in pathlib.Path("../models").rglob("*.py")]
           + [p for p in pathlib.Path("../util").rglob("*.py")]
           + [pathlib.Path("../engine.py"), pathlib.Path("../main.py")])

def identifier_hits(needle):
    found = []
    for path in SOURCES:
        src = path.read_text(encoding="utf-8")
        try:
            toks = list(tokenize.generate_tokens(io.StringIO(src).readline))
        except (tokenize.TokenError, IndentationError):
            continue
        for tok in toks:
            if tok.type == tokenize.NAME and needle in tok.string.lower():
                found.append(f"{path}:{tok.start[0]}: {tok.string}")
    return found

print("searching identifiers only (not comments, docstrings or printed text)\n")
for pat in ["nms", "non_max", "anchor", "proposal", "iou_threshold"]:
    hits = identifier_hits(pat)
    print(f"  {pat:<14} -> {len(hits)} identifier(s)")
    for h in hits[:4]:
        print("        ", h)

print("\nNothing. No non-maximum suppression, no anchors, no proposals anywhere in")
print("the pipeline -- the boxes you plotted above are the raw model output.")

**Zero** hits for `nms`, `non_max`, `anchor`, and `iou_threshold`. The single `proposal` hit is the word appearing in a docstring in [`models/detr.py`](../models/detr.py#L93) — there is no proposal stage in the code.

> *"DETR simplifies the detection pipeline by dropping multiple hand-designed components that encode prior knowledge, like spatial anchors or non-maximal suppression."* — §1

### See what NMS is actually *for*

An absence is hard to appreciate. So let's look at a detector that **does** need NMS — `torchvision`'s Faster R-CNN — and simply switch NMS off.

Its `roi_heads.nms_thresh` is the IoU threshold for suppression. Setting it to `1.0` means "never suppress anything", revealing the raw output that NMS normally hides.

*(First run downloads ~160 MB of Faster R-CNN weights.)*

In [ ]:
import torchvision.transforms as T
from torchvision.models.detection import (fasterrcnn_resnet50_fpn,
                                          FasterRCNN_ResNet50_FPN_Weights)

frcnn = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT).eval()
x_frcnn = T.ToTensor()(im)

results = {}
for nms_thresh, label in [(1.0, "NMS OFF"), (0.5, "NMS ON (default)")]:
    frcnn.roi_heads.nms_thresh = nms_thresh
    frcnn.roi_heads.score_thresh = 0.7          # same confidence bar as our DETR run
    with torch.no_grad():
        results[label] = frcnn([x_frcnn])[0]
    print(f"Faster R-CNN, {label:<18} -> {len(results[label]['boxes']):3d} boxes")

print(f"DETR (has no NMS at all)        -> {len(boxes):3d} boxes")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (label, r) in zip(axes, results.items()):
    ax.imshow(im); ax.axis("off")
    for bb in r["boxes"]:
        x0, y0, x1, y1 = bb.tolist()
        ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False,
                                   color="red", linewidth=1.5, alpha=.7))
    ax.set_title(f"Faster R-CNN — {label}\n{len(r['boxes'])} boxes", fontsize=11)

plot_results(im, probs, boxes, ax=axes[2], title=f"DETR — raw output\n{len(boxes)} boxes")
plt.tight_layout(); plt.show()

Look at the left panel. Without NMS, Faster R-CNN stacks a pile of near-identical boxes on each object, because **many anchors** legitimately overlap the same thing and each is scored independently. Nothing inside the model prevents that — NMS (middle) is a separate, non-differentiable script that sorts by score and deletes overlaps.

DETR (right) never generates the mess in the first place. Its bipartite matching loss (notebook `05`) makes a duplicate prediction *cost more*, so the model is trained not to produce one.

**Deduplication moved out of post-processing and into the loss function.** That's the sentence to remember from this notebook.

## 5. What "a set of 100 predictions" really looks like

We threw away 95 of the 100 queries above. Let's look at all of them. Each query produces 92 logits; the **last** one is `∅` (no object).

In [ ]:
logits = outputs["pred_logits"][0].cpu()       # shape: (100, 92)  = (queries, classes+1)
pred_boxes = outputs["pred_boxes"][0].cpu()    # shape: (100, 4)   = (queries, cxcywh)
print("pred_logits:", tuple(logits.shape), " pred_boxes:", tuple(pred_boxes.shape))

all_probs = logits.softmax(-1)                 # (queries, classes+1) = (100, 92)
p_noobj = all_probs[:, -1]                     # shape: (100,)  probability of "no object"

print(f"\nqueries confidently predicting ∅ (p > 0.9): {(p_noobj > 0.9).sum().item()} / 100")
print(f"queries confidently predicting an object  : {(all_probs[:, :-1].max(-1).values > 0.9).sum().item()} / 100")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
order = p_noobj.argsort()                      # sort queries by P(no object)
ax.bar(range(100), p_noobj[order], color="#bbb", label="P(∅ = no object)")
ax.bar(range(100), all_probs[:, :-1].max(-1).values[order],
       color="#0b72b9", label="P(best real class)")
ax.set_xlabel("query slot (sorted)"); ax.set_ylabel("probability")
ax.set_title("All 100 predictions: a handful of objects, 95 confident ∅")
ax.legend(); plt.tight_layout(); plt.show()

This is the shape of *set prediction*: DETR always emits exactly 100 slots, and learns to mark the unused ones `∅`. The paper sets N=100 because it must be *"significantly larger than the typical number of objects in an image"* (§3.1) — COCO images average ~7 objects, max 63.

### Boxes are normalized `(cx, cy, w, h)`

An easy thing to get wrong. DETR does **not** predict pixels — it predicts *fractions of the image*, in center format, squashed through a sigmoid.

In [ ]:
b = pred_boxes[keep][0]                 # one detected box, shape: (4,)
print("raw model output (cx, cy, w, h), normalized to [0,1]:")
print("   ", [round(v, 3) for v in b.tolist()])
print("   -> all values in [0,1]? ", bool(((b >= 0) & (b <= 1)).all()))

print("\nconverted to absolute pixel corners (x0, y0, x1, y1):")
print("   ", [round(v, 1) for v in rescale_bboxes(b[None], im.size)[0].tolist()])
print(f"    (image is W={im.size[0]}, H={im.size[1]})")

Why normalized center-format?

- **Normalized** → the same model handles any input resolution. (DETR resizes the shortest side to 800px but never crops to a fixed square.)
- **Center-format** → `sigmoid()` maps directly onto it, so every one of the 4 numbers is naturally bounded in `[0,1]`. Look at [`models/detr.py:68`](../models/detr.py#L68): `self.bbox_embed(hs).sigmoid()`.

Conversion to pixel corners happens in `PostProcess` ([`models/detr.py:258`](../models/detr.py#L258)), *outside* the model.

## 6. The whole model in one glance

In [ ]:
print("DETR.forward -- exactly the model we defined in section 0:\n")
print("  images                      shape: (B, 3, H, W)  + padding mask (B, H, W)")
print("      |")
print("      v  self.backbone[0]       ResNet-50, frozen BatchNorm")
print("  features                    shape: (B, 2048, H/32, W/32)")
print("      |                        (the mask is downsampled to match)")
print("      v  self.backbone[1]       sine positional encoding")
print("      v  self.input_proj        1x1 Conv2d(2048 -> 256)")
print("  projected                   shape: (B, 256, H/32, W/32)")
print("      |")
print("      v  self.transformer       6 encoder + 6 decoder layers")
print("  hs                          shape: (6, B, 100, 256)   <- one per decoder layer")
print("      |")
print("      +--> self.class_embed     Linear(256 -> 92)   shape: (B, 100, 92)")
print("      +--> self.bbox_embed      MLP(256->256->256->4).sigmoid()")
print("                                                    shape: (B, 100, 4)")
print()
print("input_proj :", model.input_proj)
print("query_embed:", model.query_embed, "  <- the 100 learned object queries")

## What's next

You've seen *that* it works. The next notebooks explain *why*:

| Notebook | Question it answers |
|---|---|
| `01` | *(prerequisite)* What is attention, really? Build it from scratch |
| `03` | How does a 640×480 photo become a sequence a transformer can read? |
| `04` | What *is* an "object query", and how does the decoder turn 100 of them into 100 boxes? |
| `05` | How can you compute a loss on an unordered set? (the Hungarian matcher) |
| `06` | What is the model actually looking at? (attention maps from the paper) |
| `07` | Train DETR yourself on a tiny dataset and watch the matching happen |
| `08` | **Capstone** — write DETR from scratch in 50 lines and load the real weights |

## Exercises

**Exercise 1.** Run detection on `load_image("street")`. How many objects, and how many queries stayed `∅`?

<details><summary>Solution</summary>

```python
im2 = load_image("street")
p2, b2, o2, k2 = detect(model, im2, device, threshold=0.9)
print(f"{k2.sum().item()} detections, {100 - k2.sum().item()} queries said no-object")
plot_results(im2, p2, b2); plt.show()
```

A busier scene uses more of the 100 slots, but still nowhere near all of them. N=100 is chosen to comfortably exceed COCO's worst case (63 objects).
</details>

---

**Exercise 2.** Drop the threshold to 0.5. Do duplicate boxes appear on the same object?

<details><summary>Solution</summary>

```python
p3, b3, _, k3 = detect(model, im, device, threshold=0.5)
print(f"threshold 0.9 -> {len(boxes)} boxes;  threshold 0.5 -> {len(b3)} boxes")
plot_results(im, p3, b3); plt.show()
```

You get a few more low-confidence boxes, but they are mostly *different* objects or slight variants — not the stack of near-identical copies that Faster R-CNN produced with NMS off. Duplicate suppression is already baked in.
</details>

---

**Exercise 3.** Set `threshold=0.0` and plot all 100 boxes. Where do the `∅` boxes sit?

<details><summary>Solution</summary>

```python
allb = rescale_bboxes(outputs["pred_boxes"][0].cpu(), im.size)
fig, ax = plt.subplots(figsize=(9, 7)); ax.imshow(im); ax.axis("off")
for bb in allb:
    x0, y0, x1, y1 = bb.tolist()
    ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, color="c", lw=.8, alpha=.5))
plt.show()
```

Many `∅` boxes are large and centered, or clustered in a corner. They are the "default guesses" of query slots that found nothing — the specialization you'll plot properly in notebook `04` §6.
</details>

---

**Exercise 4.** `class_embed` outputs 92 numbers but COCO has 80 real categories. Explain both discrepancies.

<details><summary>Solution</summary>

- **80 → 91:** COCO's category *ids* are not contiguous — they run 1..90 with gaps left by removed categories. DETR indexes by raw id, so it needs 91 slots and fills gaps with `N/A`. See the comment at [`models/detr.py:304`](../models/detr.py#L304).
- **91 → 92:** the extra slot is `∅`, the no-object class, which is what lets a fixed 100 predictions describe an image with 4 objects.

Getting this off by one silently mislabels every class after the gap.
</details>